In [3]:
!pip install FinanceDataReader, pandas

'pip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [6]:
import FinanceDataReader as fdr
import pandas as pd
from datetime import datetime, timedelta
import time
import os

# ===== 설정 =====
END_DATE = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d')
OUTPUT_DIR = 'stock_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"수집 기간: {START_DATE} ~ {END_DATE}")

# ===== 1. 한국 전체 종목 리스트 가져오기 =====
print("종목 리스트 수집 중...")
kospi = fdr.StockListing('KOSPI')[['Code', 'Name']]
kosdaq = fdr.StockListing('KOSDAQ')[['Code', 'Name']]
kospi['Market'] = 'KOSPI'
kosdaq['Market'] = 'KOSDAQ'

all_stocks = pd.concat([kospi, kosdaq], ignore_index=True)
print(f"총 종목 수: {len(all_stocks)}개")

# ===== 2. 종목별 OHLCV 수집 =====
failed = []
collected = []

for i, row in all_stocks.iterrows():
    code = row['Code']
    name = row['Name']
    market = row['Market']

    try:
        df = fdr.DataReader(code, START_DATE, END_DATE)

        if df.empty:
            print(f"  [SKIP] {code} {name} - 데이터 없음")
            continue

        df['Code'] = code
        df['Name'] = name
        df['Market'] = market
        df.index.name = 'Date'

        # 종목별 CSV 저장
        save_path = os.path.join(OUTPUT_DIR, f"{code}_{name}.csv")
        df.to_csv(save_path, encoding='utf-8-sig')
        collected.append(row)

        if (i + 1) % 100 == 0:
            print(f"  진행 중: {i+1}/{len(all_stocks)} ({code} {name})")

        time.sleep(0.3)  # 서버 부하 방지

    except Exception as e:
        print(f"  [ERROR] {code} {name}: {e}")
        failed.append({'Code': code, 'Name': name, 'Error': str(e)})
        time.sleep(1)

# ===== 3. 결과 저장 =====
# 수집 성공 목록
collected_df = pd.DataFrame(collected)
collected_df.to_csv('collected_list.csv', index=False, encoding='utf-8-sig')

# 실패 목록
if failed:
    failed_df = pd.DataFrame(failed)
    failed_df.to_csv('failed_list.csv', index=False, encoding='utf-8-sig')
    print(f"\n실패 종목: {len(failed)}개 → failed_list.csv 저장됨")

print(f"\n✅ 완료! 성공: {len(collected)}개 / 전체: {len(all_stocks)}개")
print(f"저장 위치: {OUTPUT_DIR}/")

수집 기간: 2021-04-07 ~ 2026-04-06
종목 리스트 수집 중...
총 종목 수: 2773개
  진행 중: 100/2773 (088980 맥쿼리인프라)
  진행 중: 200/2773 (073240 금호타이어)
  진행 중: 300/2773 (017940 E1)
  진행 중: 400/2773 (002810 삼영무역)
  진행 중: 500/2773 (010580 에스엠벡셀)
  진행 중: 600/2773 (021820 세원정공)
  진행 중: 700/2773 (068290 삼성출판사)
  진행 중: 800/2773 (023960 에쓰씨엔지니어링)
  진행 중: 900/2773 (009440 KC그린홀딩스)
  진행 중: 1000/2773 (031980 피에스케이홀딩스)
  진행 중: 1100/2773 (086900 메디톡스)
  진행 중: 1200/2773 (220100 퓨쳐켐)
  진행 중: 1300/2773 (416180 신성에스티)
  진행 중: 1400/2773 (394420 리센스메디컬)
  진행 중: 1500/2773 (452450 피아이이)
  진행 중: 1600/2773 (263720 디앤씨미디어)
  진행 중: 1700/2773 (038540 상상인)
  진행 중: 1800/2773 (246720 아스타)
  진행 중: 1900/2773 (478560 블랙야크아이앤씨)
  진행 중: 2000/2773 (095910 에스에너지)
  진행 중: 2100/2773 (088910 동우팜투테이블)
  진행 중: 2200/2773 (363250 진시스템)
  진행 중: 2300/2773 (290090 트윔)
  진행 중: 2400/2773 (075130 플랜티넷)
  진행 중: 2500/2773 (052300 오션인더블유)
  진행 중: 2600/2773 (130740 티피씨글로벌)
  진행 중: 2700/2773 (0099X0 IBKS제25호스팩)

✅ 완료! 성공: 2773개 / 전체: 2773개
저장 위치: stock_data/
